In [ ]:
import numpy as np
import filters_exp as exp
from matplotlib import pyplot as plt
from numba import njit
from numba_progress import ProgressBar

from filters import (
    # parameter dtypes
    std_env_parameters, laplace_env_parameters,
    GMVC_parameters, NLMS_parameters, LMLS_parameters, MCC_parameters, RZA_LMS_parameters,
    sKF_parameters, sKF_L_parameters,
    # signal / environment helpers
    autocorr_matrix_calc, autocorr_matrix_estimate, AR_settling_time, 
    std_gaussian_behavior, laplace_noise_behavior,
    # algorithms
    NLMS_algorithm, GMVC_algorithm, LMLS_algorithm, MCC_algorithm, RZA_LMS_algorithm,
    sKF_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm,
    sKF_integral_algorithm, sKF_L_integral_algorithm,
    # monte carlo driver
    MC_Simulations,
)

# Matplotlib configuration
%config InlineBackend.figure_format = 'svg'

# General Parameters

In [ ]:
NR = 256
num_workers = 4
num_chunks = 16

N = int(50e3)
L = 64
ho = np.sinc(np.linspace(0,1,L))
ho = ho/np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3 # Gaussian Noise
scale_v = np.sqrt(1e-3) # Laplacian Noise

AR = np.array([1.0, 0.0])
#AR = np.array([1.0, -0.6, 0.85])
#AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

# Algorithm Parameters
mu = 0.1
delta = 1e-3

mu_lmls = 1.4*mu/(L*var_x)
a_lmls = 2

mu_mcc = 1.4*mu/(L*var_x)
sigma_mcc = 1

epsilon_gaussian = 5e-9
epsilon_laplacian_1 = 6.5e-9
epsilon_laplacian_2 = 2e-8
var_tilde_0 = 10000*np.linalg.norm(ho)**2/L

In [ ]:
np.linalg.norm(ho)**2/L

In [ ]:
%matplotlib ipympl

plt.figure()
plt.plot(ho)
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.title('System to be identified')
plt.show()

# Gaussian Noise

In [ ]:
NLMS_Parameters = NLMS_parameters(label="NLMS", mu=mu, delta=delta)
LMLS_Parameters = LMLS_parameters(label="LMLS", mu=mu_lmls, a=a_lmls)
MCC_Parameters  = MCC_parameters(label='MCC', mu=mu_mcc, sigma=sigma_mcc)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF - Laplacian", epsilon=epsilon_laplacian_1, b_eta=np.sqrt(var_v/2), v_tilde_0=var_tilde_0)

env_parameters = std_env_parameters(ho=ho, AR=AR, var_v=var_v, var_x=var_x)

Algorithms = (NLMS_algorithm, LMLS_algorithm, MCC_algorithm, sKF_algorithm, sKF_L_algorithm)
Alg_Parameters = (NLMS_Parameters, LMLS_Parameters, MCC_Parameters, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters)
#Algorithms = (NLMS_algorithm, sKF_L_exact_algorithm)
#Alg_Parameters = (NLMS_Parameters, sKF_Laplacian_Parameters)

Gaussian_MC_measures = exp.dask_MC_Simulations(N, 
                                        NR, 
                                        env_parameters,
                                        std_gaussian_behavior,
                                        Algorithms,
                                        Alg_Parameters,
                                        h0,
                                        num_workers = num_workers,
                                        num_chunks = num_chunks)

In [ ]:
%matplotlib ipympl

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

plt.figure(figsize=(18, 6))
plt.subplot(1,2,1)
k=0
for params in Alg_Parameters:
    label = params.label
    plt.plot(10*np.log10(Gaussian_MC_measures[label]['MSD']), label=label, color=colors[k])
    k += 1
plt.ylabel("MSD (dB)")
plt.xlabel("Iterations")
plt.title("Gaussian Noise")
plt.legend()

plt.subplot(1,2,2)
k=0
for params in Alg_Parameters:
    label = params.label
    plt.semilogy(Gaussian_MC_measures[label]['var'], label=label, color=colors[k])
    k += 1
plt.ylabel("v_tilde")
plt.xlabel("Iterations")
plt.title("Gaussian Noise")
plt.tight_layout()
plt.show()

# Laplacian Noise

In [ ]:
NLMS_Parameters = NLMS_parameters(label="NLMS", mu=mu, delta=delta)
LMLS_Parameters = LMLS_parameters(label="LMLS", mu=mu_lmls, a=a_lmls)
MCC_Parameters  = MCC_parameters(label='MCC', mu=mu_mcc, sigma=sigma_mcc)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF - Laplacian", epsilon=epsilon_laplacian_2, b_eta=np.sqrt(var_v/2), v_tilde_0=var_tilde_0)

env_parameters = laplace_env_parameters(ho=ho, AR=AR, scale_v=scale_v, var_x=var_x)

Algorithms = (NLMS_algorithm, LMLS_algorithm, MCC_algorithm, sKF_algorithm, sKF_L_algorithm)
Alg_Parameters = (NLMS_Parameters, LMLS_Parameters, MCC_Parameters, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters)
#Algorithms = (NLMS_algorithm, sKF_L_exact_algorithm)
#Alg_Parameters = (NLMS_Parameters, sKF_Laplacian_Parameters)

laplacian_MC_measures = exp.dask_MC_Simulations(N, 
                                        NR, 
                                        env_parameters,
                                        laplace_noise_behavior,
                                        Algorithms,
                                        Alg_Parameters,
                                        h0,
                                        num_workers = num_workers,
                                        num_chunks = num_chunks)

In [ ]:
Alg_Parameters

In [ ]:
%matplotlib ipympl

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

plt.figure(figsize=(18, 6))
plt.subplot(1,2,1)
k=0
for params in Alg_Parameters:
    label = params.label
    plt.plot(10*np.log10(laplacian_MC_measures[label]['MSD']), label=label, color=colors[k])
    k += 1
plt.ylabel("MSD (dB)")
plt.xlabel("Iterations")
plt.title("Laplacian Noise")
plt.legend()

plt.subplot(1,2,2)
k=0
for params in Alg_Parameters:
    label = params.label
    plt.semilogy(laplacian_MC_measures[label]['var'], label=label, color=colors[k])
    k += 1
plt.ylabel("v_tilde")
plt.xlabel("Iterations")
plt.title("Laplacian Noise")
plt.tight_layout()
plt.show()